**Imports and Model Setup**

In [71]:
import torch

print("PyTorch version:", torch.__version__)  # Should show CUDA version
print("CUDA version in PyTorch:", torch.version.cuda)  # Should match installed CUDA (e.g., 12.1)
print("CUDA available:", torch.cuda.is_available())  # Should return True
print("Number of GPUs:", torch.cuda.device_count())  # Should be > 0
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

PyTorch version: 2.5.1+cu121
CUDA version in PyTorch: 12.1
CUDA available: True
Number of GPUs: 1
GPU Name: NVIDIA GeForce RTX 3060


In [72]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import cv2
import torch

In [73]:
from mobile_sam import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

model_type = "vit_t"
sam_checkpoint = "weights/mobile_sam.pt"
device = "cpu"  # set device to cpu temporarily for dataset transforms

mobile_sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
mobile_sam.to(device=device)

predictor = SamPredictor(mobile_sam)
mask_generator = SamAutomaticMaskGenerator(mobile_sam)

c:\Users\user\Documents\slurry_viscnet\MobileSAM_Vortex\mobile_sam\build_sam.py:91: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


**Generate Dataset**

## Data

Preparing Data for training/validation (not yet test). Training and validation process using PyTorch requires two main implementations: `Dataset`, `Dataloader`.  

In [87]:
import os.path as osp
import numpy as np
import cv2
from patchify import patchify
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import namedtuple

ImageMaskPathItem = namedtuple("ImageMaskPathItem", ["image_path", "mask_path", "box_path"])

class ImageMaskDataset(Dataset):
    def __init__(self, root, image_subdir, mask_subdir, box_subdir, transform=None):
        super().__init__()
        self.root = root
        self.image_subdir = image_subdir
        self.mask_subdir = mask_subdir
        self.box_subdir = box_subdir
        self.transform = transform

        images = glob.glob(osp.join(root, image_subdir, "*.png"))
        images = sorted(images)
        masks = [image.replace(image_subdir, mask_subdir) for image in images]
        boxes = [mask.replace(mask_subdir, box_subdir).replace(".png", ".npy") for mask in masks]

        self.path_items = [ImageMaskPathItem(image_path=image, mask_path=mask, box_path=box) for image, mask, box in zip(images, masks, boxes)]
        self._sanity_check()
        print(f"Found {len(self.path_items)} image-mask pairs")

    def _sanity_check(self):
        for item in self.path_items:
            assert osp.exists(item.image_path), f"Image path {item.image_path} does not exist"
            assert osp.exists(item.mask_path), f"Mask path {item.mask_path} does not exist"
            assert osp.exists(item.box_path), f"Mask path {item.box_path} does not exist"

    def __getitem__(self, index):
        item = self.path_items[index]
        image = cv2.imread(item.image_path)
        mask = cv2.imread(item.mask_path, cv2.IMREAD_GRAYSCALE)
        box = np.load(item.box_path)
        
        if self.transform:
            transformed = self.transform(image=image, mask=mask) #normalized and permuted
            image = transformed["image"]
            mask = transformed["mask"]
            box = torch.tensor(box)
            

        return image, mask, box

    def __len__(self):
        return len(self.path_items)

In [ ]:
import torch
from mobile_sam.utils import transforms
from torchvision import transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2

transform = A.Compose(
    [
        A.Normalize(max_pixel_value=255.0),
        ToTensorV2(), #equivalent of permute/transpose
    ]
)

ds = ImageMaskDataset(root="processed_data", image_subdir="images", mask_subdir="masks", box_subdir="boxes", transform=transform)
train_ds, val_ds = train_test_split(ds, test_size=0.2, random_state=42)
print(f"Train dataset size: {len(train_ds)}")
print(f"Validation dataset size: {len(val_ds)}")

print("Train dataset sample:")
print(train_ds[0][0].shape)
print(train_ds[0][1].shape)
print(train_ds[0][2].shape)

Found 17237 image-mask pairs
Train dataset size: 13789
Validation dataset size: 3448
Train dataset sample:
torch.Size([3, 256, 256])
torch.Size([3, 256, 256])
torch.Size([256, 256])


In [110]:
print(train_ds[0][0].shape)
print(train_ds[0][1].shape)
print(train_ds[0][2].shape)

torch.Size([3, 256, 256])
torch.Size([256, 256])
torch.Size([4])


In [ ]:
BATCH_SIZE = 8
NUM_WORKERS = 4

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [102]:

print(train_dl[0][1].shape)
print(train_dl[0][2].shape)

TypeError: 'DataLoader' object is not subscriptable

**Dataset Processor for MobileSAM**

In [90]:
from mobile_sam import sam_model_registry, SamPredictor

model_type = "vit_t"
sam_checkpoint = "model/MobileSAM_Vortex_checkpoint.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

mobile_sam = sam_model_registry["vit_t"](checkpoint="weights/mobile_sam.pt")
mobile_sam.to(device=device)
predictor = SamPredictor(mobile_sam)

c:\Users\user\Documents\slurry_viscnet\MobileSAM_Vortex\mobile_sam\build_sam.py:91: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


In [91]:
# Freeze layers in MobileSAM
for name, param in mobile_sam.named_parameters():
    if name.startswith("image_encoder"):  # Assuming "image_encoder" corresponds to the vision encoder
        param.requires_grad = False  # Freeze the vision encoder
    elif name.startswith("prompt_encoder"):  # Assuming "prompt_encoder" exists in MobileSAM
        param.requires_grad = False  # Freeze the prompt encoder

# Verify trainable parameters
trainable_params = sum(p.numel() for p in mobile_sam.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

# Iterate through all parameters and check their trainable status
print("MobileSAM Parameters:")
for name, param in mobile_sam.named_parameters():
    print(f"{name}: {'Trainable' if param.requires_grad else 'Frozen'}")

# Count the total trainable parameters
trainable_params = sum(p.numel() for p in mobile_sam.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters in MobileSAM: {trainable_params}")

Trainable parameters: 4058340
MobileSAM Parameters:
image_encoder.patch_embed.seq.0.c.weight: Frozen
image_encoder.patch_embed.seq.0.bn.weight: Frozen
image_encoder.patch_embed.seq.0.bn.bias: Frozen
image_encoder.patch_embed.seq.2.c.weight: Frozen
image_encoder.patch_embed.seq.2.bn.weight: Frozen
image_encoder.patch_embed.seq.2.bn.bias: Frozen
image_encoder.layers.0.blocks.0.conv1.c.weight: Frozen
image_encoder.layers.0.blocks.0.conv1.bn.weight: Frozen
image_encoder.layers.0.blocks.0.conv1.bn.bias: Frozen
image_encoder.layers.0.blocks.0.conv2.c.weight: Frozen
image_encoder.layers.0.blocks.0.conv2.bn.weight: Frozen
image_encoder.layers.0.blocks.0.conv2.bn.bias: Frozen
image_encoder.layers.0.blocks.0.conv3.c.weight: Frozen
image_encoder.layers.0.blocks.0.conv3.bn.weight: Frozen
image_encoder.layers.0.blocks.0.conv3.bn.bias: Frozen
image_encoder.layers.0.blocks.1.conv1.c.weight: Frozen
image_encoder.layers.0.blocks.1.conv1.bn.weight: Frozen
image_encoder.layers.0.blocks.1.conv1.bn.bias: F

**Training Loop**

In [92]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Initialize the optimizer for the mask decoder
optimizer = Adam(mobile_sam.mask_decoder.parameters(), lr=1e-3, weight_decay=0)

from torch import nn
from torch.nn import functional as F

class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()

    def forward(self, inputs, targets, smooth=1):

        # comment out if your model contains a sigmoid or equivalent activation layer
        inputs = F.sigmoid(inputs)

        # flatten label and prediction tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2.0 * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
        BCE = F.binary_cross_entropy(inputs, targets, reduction="mean")
        Dice_BCE = BCE + dice_loss

        return Dice_BCE

#define loss and scheduler
seg_loss = DiceBCELoss()
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=3, verbose=True)

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [111]:
from tqdm import tqdm
from statistics import mean
import torch

# Training loop
num_epochs = 25
device = "cuda" if torch.cuda.is_available() else "cpu"
mobile_sam.to(device)

mobile_sam.train()
for epoch in range(num_epochs):
    train_losses = []
    for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{num_epochs} - Training"):
        # stacking mask/training model/mask retrieve
        images, masks, box = batch
        images = images.to(device)
        masks = masks.to(device)
        box = box.to(device)
    
        batched_input = [{"image": image[idx], "mask_inputs" : masks[idx], "boxes" : masks[idx], "original_size": (256, 256)} for idx in len(batch)]  
        outputs = mobile_sam(batched_input=batched_input, multimask_output=False)  # requires the whole batch
        predicted_masks = torch.stack([output["masks"].squeeze(1).float() for output in outputs])

        # loss calculation/optimization
        train_loss = seg_loss(predicted_masks, masks)
        train_losses.append(train_loss.item())
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        if (len(train_losses)) % 100 == 0:
            print(f"Batch {len(train_losses)+1}/{len(train_dl)} - Training Loss: {train_loss.item():.2f}")

    mean_train_loss = mean(train_losses)
    train_losses.clear()

    # Validation loss calculation
    mobile_sam.eval()
    val_losses = []
    with torch.no_grad():
        for batch in tqdm(val_dl, desc=f"Epoch {epoch+1}/{num_epochs} - Validation"):
            images, masks = batch
            images = images.to(device)
            masks = (masks / 255).round().float().to(device)
            # mask_batch = torch.stack([sample["ground_truth_mask"].unsqueeze(0).to(device) for sample in batch_group])
            batched_input = [{"image": img, "original_size": (256, 256)} for img in images]  # FIXME: Check if this is the right way to pass the input
            outputs = mobile_sam(batched_input=batched_input, multimask_output=False)
            predicted_masks = torch.stack([output["masks"].squeeze(1).float() for output in outputs])
            val_loss = seg_loss(predicted_masks, masks)
            val_losses.append(val_loss.item())
    mean_val_loss = mean(val_losses)

    print(f"EPOCH: {epoch + 1}/{num_epochs} - Train Loss: {mean_train_loss:.4f} | Val Loss: {mean_val_loss:.4f}")
    scheduler.step(val_loss)
    mobile_sam.train()

Epoch 1/25 - Training:   0%|          | 0/3448 [00:07<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# Save the state_dict after moving layers
torch.save(mobile_sam.state_dict(), "model/MobileSAM_Vortex_checkpoint.pth")

## Inference

In [ ]:
from mobile_sam import sam_model_registry, SamPredictor

vortex_model = sam_model_registry["vit_t"](checkpoint="model/MobileSAM_Vortex_checkpoint.pth")
vortex_model.eval()
predictor = SamPredictor(vortex_model)

In [ ]:
import random

test_number = 3

for i in range(test_number):
    idx = random.randint(0, len(val_ds) - 1)

    test_image = val_ds[idx].cpu().numpy()
    test_image = np.transpose(test_image, (1, 2, 0))
    test_box = test_boxes[idx].cpu().numpy()
    test_mask = test_masks[idx].cpu().numpy()

    predictor.set_image(test_image)
    masks, _, _ = predictor.predict(box=test_box, return_logits=True)

    masks = np.transpose(masks, (1, 2, 0))
    masks = masks.astype(np.uint8)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(test_image)
    axes[0].set_title("Original Image")

    axes[1].imshow(test_mask, cmap="gray")
    axes[1].set_title("Ground Truth Mask")

    axes[2].imshow(masks)
    axes[2].set_title("Inference Output")

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xticklabels([])
        ax.set_yticklabels([])

    plt.show()

In [ ]:
torch.cuda.empty_cache()